In [1]:
# Instalação opcional — execute somente se o ambiente não tiver as dependências.
# No Colab ou em um kernel novo, descomente a linha abaixo:
%pip install -q librosa
print("Dependência opcional: librosa.")

Note: you may need to restart the kernel to use updated packages.
Dependência opcional: librosa.


# 🎵 Geração de Fingerprints Espectrais para Reconhecimento de Emoção em Música

## 1. Visão Geral e Configuração
Este notebook implementa um sistema de **fingerprinting espectral híbrido** (Percentil + Constelação) para análise de emoção em áudio. O sistema identifica as frequências mais evidentes no top 30 global e resume, por contagem, quais bandas aparecem com mais força em cada bloco.

### 📊 Base de Dados e Metodologia
- **Dataset**: DEAM (Database for Emotion Analysis using Multimodal signals).
- **Entrada**: Arquivos de áudio originais (.mp3, .wav).
- **Anotações**: Valência e Arousal sincronizados a cada 500ms.
- **Técnica**: Extração de picos de magnitude via percentil adaptativo, com ranking absoluto top 30 e resumo das bandas mais representadas.

In [2]:
#  IMPORTAÇÕES E CONFIGURAÇÃO GERAL
import os, re, io, zlib, math, time, warnings, pathlib
import numpy as np
import pandas as pd
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.express as px
except Exception:
    go = None
    make_subplots = None
    px = None

try:
    import librosa
except Exception as exc:
    raise ImportError(
        "Falha ao importar librosa. Instale a dependência no ambiente antes de executar o pipeline."
    ) from exc
import unicodedata, random
from typing import List, Tuple, Dict, Any, Optional
from dataclasses import dataclass

warnings.filterwarnings("ignore")

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

try:
    from scipy.ndimage import median_filter
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

print("✅ Importações carregadas com sucesso")

✅ Importações carregadas com sucesso


In [3]:
# Execucao local: nenhum Google Drive precisa ser montado.
# Esta celula fica inofensiva caso o notebook seja aberto fora do ambiente local.
print("Execucao local configurada; Google Drive nao e necessario.")


Execucao local configurada; Google Drive nao e necessario.


In [4]:
# ============================================================================
# CONFIGURACOES GLOBAIS PADRONIZADAS PARA EXPERIMENTOS DE FINGERPRINTING
# ============================================================================

import json
import pathlib
from datetime import datetime

# Configuracoes de processamento de audio
RANDOM_STATE = 42

# Segmentacao de blocos — alinhado com as anotacoes DEAM (sample_15000ms)
AUDIO_START_SEC        = 15.0   # inicio do processamento
BLOCK_SIZE_SEC         = 10.0   # duracao de cada bloco
BLOCK_STEP_SEC         = 10.0   # passo entre blocos (sem sobreposicao)
MIN_BLOCK_DURATION_SEC = 5.0    # descarta blocos menores que este valor

# Alias de compatibilidade com funcoes legadas
SKIP_SECONDS = AUDIO_START_SEC

TOP_K = 30
N_FFT_GLOBAL = 2048
HOP_LENGTH_GLOBAL = 1024
WINDOW_TYPE = "hann"
SR_TARGET = 44100

# Limite de processamento (0 = processar tudo)
MAX_FILES_TO_PROCESS = 0

# Metodo do fingerprint
FINGERPRINT_METHOD = "top30_rank_absolute"
NYQUIST = SR_TARGET / 2

FREQ_BANDS = {
    "low":       (20, 250),
    "mid":       (250, 2000),
    "high":      (2000, 8000),
    "very_high": (8000, NYQUIST),
}

# Colunas de target
TARGET_COLUMNS = ["valence", "arousal"]
GROUP_COLUMN = "song_id"

# Diretorio raiz local do dataset
BASE_RESULTS_DIR = pathlib.Path(r"C:\dev\python\TCC Ajustado\Dados")
FINGERPRINT_DIR  = BASE_RESULTS_DIR / "fingerprint_rank"
FINGERPRINT_DIR.mkdir(parents=True, exist_ok=True)

# Aliases mantidos para compatibilidade com funcoes existentes
OUTPUTS_DIR          = FINGERPRINT_DIR
RESULTS_DIR          = FINGERPRINT_DIR
TABLES_DIR           = FINGERPRINT_DIR
FIGURES_DIR          = FINGERPRINT_DIR
SUMMARY_BY_SONG_DIR  = FINGERPRINT_DIR
DETAIL_BY_SONG_DIR   = FINGERPRINT_DIR
TEMPORAL_BY_SONG_DIR = FINGERPRINT_DIR

# Nome do arquivo de saida unico
OUTPUT_PARQUET_NAME = "fingerprint_rank.parquet"

# Configuracoes de exportacao
EXPORT_FORMATS = {
    "parquet": True,
    "csv": False,
    "xlsx": False,
    "html": False,
    "png": None,
}

try:
    import kaleido
    EXPORT_FORMATS["png"] = False
    KALEIDO_AVAILABLE = True
except ImportError:
    KALEIDO_AVAILABLE = False

# Configuracoes de visualizacao
PLOT_HEIGHT = 600
PLOT_WIDTH  = 1200
TEMPLATE    = "plotly_white"


# ============================================================================
# FLAGS DE PERFORMANCE E CACHE
# ============================================================================
GENERATE_PLOTS = False
EXPORT_HTML = False
EXPORT_PNG = False
SAVE_DETAIL_EVENTS = False
SAVE_TEMPORAL_CURVES = False         # nao necessario: saida consolidada em fingerprint_rank.parquet
GENERATE_TEMPORAL_FEATURES = False   # desabilitado nesta versao consolidada
CACHE_SKIP_EXISTING = True
DOWNSAMPLE_AUDIO_TO = None

# Seguranca metodologica: preserva valence/arousal/quadrant para o notebook de modelagem.
INCLUDE_TARGETS_IN_OUTPUT = True
TARGET_COLUMNS_TO_KEEP_IN_OUTPUT = ["valence", "arousal", "quadrant", "quadrant_label"]

TARGET_LEAKAGE_COLUMNS = [
    "dist_valence_limiar", "dist_arousal_limiar", "dist_centro_va",
]

print("[OK] Configuracoes globais carregadas")
print(f"     Saida principal : {FINGERPRINT_DIR.absolute()}")
print(f"     Arquivo de saida: {OUTPUT_PARQUET_NAME}")
print(f"     Blocos: inicio={AUDIO_START_SEC}s | tamanho={BLOCK_SIZE_SEC}s | passo={BLOCK_STEP_SEC}s | minimo={MIN_BLOCK_DURATION_SEC}s")


[OK] Configuracoes globais carregadas
     Saida principal : C:\dev\python\TCC Ajustado\Dados\fingerprint_rank
     Arquivo de saida: fingerprint_rank.parquet
     Blocos: inicio=15.0s | tamanho=10.0s | passo=10.0s | minimo=5.0s


# 2. Parametros e Utilitarios

Nesta secao definimos os caminhos locais de entrada/saida, parametros de audio para o STFT e funcoes de suporte para manipulacao de arquivos.


In [5]:
# ============================================================================
# CAMINHOS DE ENTRADA
# ============================================================================

DEAM_ROOT = BASE_RESULTS_DIR
IN_PATH   = str(DEAM_ROOT / 'audio')
ANNOT_DIR = DEAM_ROOT
V_CSV     = DEAM_ROOT / 'valence.csv'
A_CSV     = DEAM_ROOT / 'arousal.csv'
OUT_ROOT  = str(FINGERPRINT_DIR)

print(f'Audio  IN : {IN_PATH}')
print(f'Valence   : {V_CSV}')
print(f'Arousal   : {A_CSV}')
print(f'Saida     : {OUT_ROOT}')

import os
ok_audio   = os.path.isdir(IN_PATH)
ok_valence = os.path.exists(V_CSV)
ok_arousal = os.path.exists(A_CSV)
print(f'  audio dir existe  : {ok_audio}')
print(f'  valence.csv existe: {ok_valence}')
print(f'  arousal.csv existe: {ok_arousal}')

try:
    if ok_valence and ok_arousal:
        df_v_raw = pd.read_csv(V_CSV)
        df_a_raw = pd.read_csv(A_CSV)

        def _melt_deam(df, name):
            m = df.melt(id_vars='song_id', var_name='ts', value_name=name)
            m['time_s'] = m['ts'].str.extract(r'(\d+)').astype(float) / 1000.0
            return m.drop(columns='ts')

        DEAM_VA = pd.merge(
            _melt_deam(df_v_raw, 'valence'),
            _melt_deam(df_a_raw, 'arousal'),
            on=['song_id', 'time_s'],
            how='inner',
        )
        # Pre-agrupa: evita varredura de 2.2M linhas por musica
        DEAM_VA_BY_SONG = DEAM_VA.groupby('song_id')
        print(f'[OK] Anotacoes V/A: {len(DEAM_VA)} pontos, {DEAM_VA["song_id"].nunique()} musicas')
        print(f'     Indexado por song_id: {DEAM_VA["song_id"].nunique()} grupos')
    else:
        DEAM_VA = None
        DEAM_VA_BY_SONG = None
        print('[WARN] valence.csv ou arousal.csv nao encontrados')
except Exception as e:
    DEAM_VA = None
    DEAM_VA_BY_SONG = None
    print(f'[ERR] Falha ao carregar V/A: {e}')

# Aliases de compatibilidade
SR         = SR_TARGET
N_FFT      = N_FFT_GLOBAL
HOP_LENGTH = HOP_LENGTH_GLOBAL

# Visual
DO_DUAL       = True
DO_OVERLAY    = True
EXPORT_HTML   = False
IMG_W, IMG_H  = 1280, 640
YLIM_HZ       = (50, 8000)
USE_LOG_Y     = True
SPEC_CMAP     = 'Inferno'
ZLIM_FIXED    = (-75, 0)
TITLE_PREFIX  = 'Fingerprint espectral - '

BANDS              = list(FREQ_BANDS.values())
TOPK_PER_BAND      = 30
TOP_K_RANK         = 30
EQUALIZE_FOR_PEAKS = True
EQ_SIZE_FREQ       = 15

AUTO_PEAK_DENSITY    = False
TARGET_PEAKS_PER_SEC = 60.0
PCTL_SEARCH_RANGE    = (80.0, 99.9)
MAX_BINSEARCH_ITERS  = 12
TOLERANCE_RELATIVE   = 0.20

SHOW_FAN_LINES = True
FAN_T_MIN      = 0.02
FAN_T_MAX      = 0.50
FAN_FANOUT     = 6

X_TICK_STEP_BASE = 2.0
PK_MARKER = dict(symbol='circle', size=6, opacity=0.95, color='#000000', line=dict(width=0.8, color='#111111'))

# Pre-computacao de valores estaticos (mesmos para todos os blocos)
_FREQS_GLOBAL = librosa.fft_frequencies(sr=SR_TARGET, n_fft=N_FFT_GLOBAL)
_FULL_BLOCK_SAMPLES = int(BLOCK_SIZE_SEC * SR_TARGET)
_FULL_BLOCK_FRAMES  = max(1, 1 + (_FULL_BLOCK_SAMPLES - N_FFT_GLOBAL) // HOP_LENGTH_GLOBAL)
_TIMES_LOCAL_FULL   = librosa.frames_to_time(
    np.arange(_FULL_BLOCK_FRAMES), sr=SR_TARGET, hop_length=HOP_LENGTH_GLOBAL
)

print('[OK] Parametros carregados')
print(f'     Pre-computacao: {_FULL_BLOCK_FRAMES} frames/bloco de {BLOCK_SIZE_SEC}s')


Audio  IN : C:\dev\python\TCC Ajustado\Dados\audio
Valence   : C:\dev\python\TCC Ajustado\Dados\valence.csv
Arousal   : C:\dev\python\TCC Ajustado\Dados\arousal.csv
Saida     : C:\dev\python\TCC Ajustado\Dados\fingerprint_rank
  audio dir existe  : True
  valence.csv existe: True
  arousal.csv existe: True
[OK] Anotacoes V/A: 2203846 pontos, 1802 musicas
     Indexado por song_id: 1802 grupos
[OK] Parametros carregados
     Pre-computacao: 429 frames/bloco de 10.0s


In [6]:
# ============================================================================
# FUNCOES DE SALVAMENTO
# ============================================================================

def save_dataframe(df, filename_base, formats=None, directory=TABLES_DIR, verbose=True):
    """Salva DataFrame em multiplos formatos configurados."""
    if formats is None:
        formats = EXPORT_FORMATS

    saved_files = {}
    directory = pathlib.Path(directory)
    directory.mkdir(parents=True, exist_ok=True)

    try:
        if formats.get("parquet"):
            fpath = directory / f"{filename_base}.parquet"
            df.to_parquet(fpath, engine="pyarrow", compression="snappy")
            saved_files["parquet"] = str(fpath)
    except Exception as e:
        print(f"Erro ao salvar Parquet: {e}")

    try:
        if formats.get("csv"):
            fpath = directory / f"{filename_base}.csv"
            df.to_csv(fpath, index=False)
            saved_files["csv"] = str(fpath)
    except Exception as e:
        print(f"Erro ao salvar CSV: {e}")

    try:
        if formats.get("xlsx"):
            fpath = directory / f"{filename_base}.xlsx"
            df.to_excel(fpath, index=False, engine="openpyxl")
            saved_files["xlsx"] = str(fpath)
    except Exception as e:
        print(f"Erro ao salvar XLSX: {e}")

    if verbose and saved_files:
        print(f"{filename_base}: salvo em {list(saved_files.keys())}")

    return saved_files

print("Funcoes de salvamento carregadas")

Funcoes de salvamento carregadas


## 2.1 Funções de Suporte (I/O e Conversão)

In [7]:
def ensure_dir(p):
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

def bytes_to_ndarray(b):
    if b is None: return np.array([])
    if isinstance(b, (bytes, bytearray)):
        try:
            raw = zlib.decompress(b)
            return np.load(io.BytesIO(raw), allow_pickle=False)
        except Exception:
            return np.load(io.BytesIO(b), allow_pickle=False)
    if isinstance(b, np.ndarray): return b
    return np.array([])

def _iter_audio_files(in_path):
    p = pathlib.Path(in_path)
    files = []
    for ext in ['*.mp3', '*.wav', '*.ogg']: files.extend(list(p.glob(ext)))
    files = sorted(files)
    out = []
    for fp in files:
        m = re.search(r'(\d+)', fp.name)
        sid = m.group(1) if m else fp.stem
        out.append((str(fp), sid))
    return out

print('Utilitarios carregados')

def assign_frequency_band(freq_hz):
    for band_name, (lo, hi) in FREQ_BANDS.items():
        if lo <= float(freq_hz) < hi:
            return band_name
    return list(FREQ_BANDS.keys())[-1]


def assign_frequency_band_vec(freq_array):
    """Versao vectorizada: retorna array de nomes de banda para um array de frequencias."""
    fa      = np.asarray(freq_array, dtype=np.float64)
    default = list(FREQ_BANDS.keys())[-1]
    result  = np.full(len(fa), default, dtype=object)
    for band_name, (lo, hi) in FREQ_BANDS.items():
        result[(fa >= lo) & (fa < hi)] = band_name
    return result


print('assign_frequency_band e assign_frequency_band_vec carregadas')


Utilitarios carregados
assign_frequency_band e assign_frequency_band_vec carregadas


# 🔊 3. Processamento Espectral Core

Aqui tratamos a normalização da escala (dB), equalização para destaque de transientes e preparação da matriz STFT.

In [8]:
def detect_scale_and_fix(S, freqs, times, zlim_fixed=ZLIM_FIXED):
    if S is None or S.size == 0: raise ValueError("Matriz S vazia")
    S = np.array(S, copy=False)
    try:
        if freqs.size and times.size and S.shape[0] == times.size and S.shape[1] == freqs.size: S = S.T
    except Exception: pass
    S = np.where(np.isfinite(S), S, np.nan)
    finite_min = float(np.nanmin(S)) if not np.isnan(S).all() else -100.0
    S = np.nan_to_num(S, nan=finite_min)
    S_min, S_max = float(np.min(S)), float(np.max(S))
    frac_neg = float((S < 0).mean()) if S.size else 0.0
    is_db_like = ((S_max <= 5.0 and S_min < -10.0) or (frac_neg > 0.6))
    if np.iscomplexobj(S): S_abs, is_db_like = np.abs(S), False
    else: S_abs = np.abs(S)
    if is_db_like: S_db = S.astype(float, copy=False)
    else:
        ref = np.max(S_abs) if np.max(S_abs) > 0 else 1.0
        S_db = librosa.amplitude_to_db(S_abs, ref=ref)
    if freqs is None or freqs.size == 0: freqs = np.linspace(0.0, 22050.0, S_db.shape[0], dtype=float)
    if times is None or times.size == 0: times = np.arange(S_db.shape[1], dtype=float)
    zmin, zmax = zlim_fixed if zlim_fixed else np.nanpercentile(S_db, [1, 99])
    return S_db, np.asarray(freqs, float), np.asarray(times, float), (float(zmin), float(zmax))

def equalize_for_peaks(S_db):
    if not SCIPY_OK or EQ_SIZE_FREQ <= 1: return S_db
    return S_db - median_filter(S_db, size=(EQ_SIZE_FREQ, 1))

print("Funcoes de escala carregadas")

Funcoes de escala carregadas


In [9]:
# ============================================================================
# FUNCOES DE CONSTRUCAO DE DATAFRAMES (TOP-30 ABSOLUTO) — otimizado
# ============================================================================


def _freq_to_midi(freq_hz):
    if freq_hz is None: return np.nan
    freq_hz = float(freq_hz)
    if not np.isfinite(freq_hz) or freq_hz <= 0: return np.nan
    return 69.0 + 12.0 * np.log2(freq_hz / 440.0)


def _midi_to_octave(midi_value):
    if midi_value is None or not np.isfinite(midi_value): return np.nan
    return float(np.floor(midi_value / 12.0) - 1.0)


def _midi_to_pitch_class(midi_value):
    if midi_value is None or not np.isfinite(midi_value): return np.nan
    return int(np.round(midi_value)) % 12


def _midi_to_semitone(midi_value):
    if midi_value is None or not np.isfinite(midi_value): return np.nan
    return float(np.round(midi_value))


def _safe_mean_std(values):
    """Media e desvio padrao ignorando NaN — usa numpy em vez de pd.Series."""
    arr = np.asarray(values, dtype=np.float64)
    valid = arr[np.isfinite(arr)]
    if valid.size == 0:
        return np.nan, np.nan
    return float(valid.mean()), float(valid.std())


def classify_emotional_quadrant(valence, arousal):
    if pd.isna(valence) or pd.isna(arousal):
        return 'INDEFINIDO', 'Indefinido'
    valence, arousal = float(valence), float(arousal)
    if np.isclose(valence, 0.0) or np.isclose(arousal, 0.0):
        return 'NEUTRO', 'Neutro/Indefinido'
    if valence > 0 and arousal > 0: return 'Q1', 'Alegre/Energetico'
    if valence < 0 and arousal > 0: return 'Q2', 'Tenso/Raivoso'
    if valence < 0 and arousal < 0: return 'Q3', 'Triste/Melancolico'
    if valence > 0 and arousal < 0: return 'Q4', 'Calmo/Relaxado'
    return 'INDEFINIDO', 'Indefinido'


def _rank_bands_from_top_peaks(df_top_peaks, top_n=TOP_K_RANK, max_bands=4):
    if df_top_peaks is None or df_top_peaks.empty:
        return []
    source = df_top_peaks.loc[~df_top_peaks['is_octave_duplicate']].copy()
    if source.empty:
        source = df_top_peaks.copy()
    ranked = source.sort_values(
        ['magnitude_db', 'rank_in_block', 'time_sec', 'freq_hz'],
        ascending=[False, True, True, True],
    ).head(top_n)
    if ranked.empty:
        return []
    band_counts = ranked['freq_band'].value_counts()
    ranked_bands = []
    for band_name, count in band_counts.items():
        ranked_bands.append((band_name, int(count)))
        if len(ranked_bands) >= max_bands:
            break
    return ranked_bands


def _enrich_musical_columns(df):
    """Vectorizado: substitui .apply() elemento a elemento por operacoes numpy."""
    df = df.copy()
    fa = df['freq_hz'].to_numpy(dtype=np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        midi = np.where(fa > 0, 69.0 + 12.0 * np.log2(fa / 440.0), np.nan)
    df['midi']           = midi
    df['octave']         = np.where(np.isfinite(midi), np.floor(midi / 12.0) - 1.0, np.nan)
    df['pitch_class']    = np.where(np.isfinite(midi), np.round(midi) % 12, np.nan)
    df['semitone_approx']= np.where(np.isfinite(midi), np.round(midi), np.nan)
    return df


def _normalized_entropy_from_counts(counts):
    vals = np.asarray([v for v in counts if pd.notna(v) and float(v) > 0], dtype=float)
    if vals.size <= 1:
        return 0.0 if vals.size == 1 else np.nan
    p = vals / vals.sum()
    return float(-(p * np.log2(p)).sum() / np.log2(len(vals)))


def _add_block_engineered_features(row, source_events=None, duration_sec=None):
    row = dict(row)
    duration = float(duration_sec if duration_sec is not None and pd.notna(duration_sec) else row.get('duration_sec', np.nan))
    if not np.isfinite(duration) or duration <= 0:
        duration = 1.0

    if source_events is not None and isinstance(source_events, pd.DataFrame) and not source_events.empty:
        if 'pitch_class' in source_events.columns:
            pc_counts = source_events['pitch_class'].dropna().value_counts().values
            row['fp_pitch_class_entropy'] = _normalized_entropy_from_counts(pc_counts)
        else:
            row['fp_pitch_class_entropy'] = np.nan
        if 'time_sec' in source_events.columns:
            unique_times = np.sort(source_events['time_sec'].dropna().unique())
            gaps = np.diff(unique_times)
            row['fp_peak_gap_mean'] = float(np.mean(gaps)) if gaps.size else np.nan
            row['fp_peak_gap_std']  = float(np.std(gaps))  if gaps.size else np.nan
        else:
            row['fp_peak_gap_mean'] = np.nan
            row['fp_peak_gap_std']  = np.nan
    else:
        row['fp_pitch_class_entropy'] = np.nan
        row['fp_peak_gap_mean']       = np.nan
        row['fp_peak_gap_std']        = np.nan

    for base in ['freq', 'mag', 'midi', 'octave', 'semitone']:
        std_col = f'fp_{base}_std'
        val = row.get(std_col, np.nan)
        row[f'fp_{base}_stability'] = float(1.0 / (1.0 + abs(float(val)))) if pd.notna(val) else np.nan
    return row


def build_block_detail_dataframe(pk_t, pk_f, pk_m, song_id, block_idx, start_time, end_time,
                                 valence=np.nan, arousal=np.nan, method=FINGERPRINT_METHOD,
                                 quadrant=None, quadrant_label=None):
    pk_t = np.asarray(pk_t if pk_t is not None else [], dtype=float)
    pk_f = np.asarray(pk_f if pk_f is not None else [], dtype=float)
    pk_m = np.asarray(pk_m if pk_m is not None else [], dtype=float)

    base_columns = [
        'song_id', 'block_idx', 'start_time', 'end_time', 'time_sec', 'frame_idx',
        'method', 'valence', 'arousal', 'quadrant', 'quadrant_label',
        'freq_hz', 'magnitude_db', 'original_freq_hz', 'freq_octave_up_hz',
        'duplicated_freq_hz', 'is_octave_duplicate', 'source_rank', 'same_time_frame',
        'freq_band', 'source_band', 'rank_in_block',
        'midi', 'octave', 'pitch_class', 'semitone_approx',
    ]

    if pk_t.size == 0:
        return pd.DataFrame(columns=base_columns)

    detail = pd.DataFrame({
        'song_id':       song_id,
        'block_idx':     int(block_idx),
        'start_time':    float(start_time),
        'end_time':      float(end_time),
        'time_sec':      pk_t.astype(float),
        'freq_hz':       pk_f.astype(float),
        'magnitude_db':  pk_m.astype(float),
        'method':        method,
        'valence':       float(valence) if pd.notna(valence) else np.nan,
        'arousal':       float(arousal) if pd.notna(arousal) else np.nan,
        'quadrant':      quadrant,
        'quadrant_label':quadrant_label,
    })

    # frame_idx: indice do primeiro aparecimento de cada valor de tempo em pk_t
    # Vectorizado O(n log n) vs O(n^2) original
    _, frame_idx_arr = np.unique(pk_t.astype(float), return_inverse=True)
    detail['frame_idx'] = frame_idx_arr.astype(int)

    # freq_band vectorizado
    detail['freq_band']        = assign_frequency_band_vec(pk_f)
    detail['source_band']      = detail['freq_band']
    detail['original_freq_hz'] = detail['freq_hz']
    detail['freq_octave_up_hz']= detail['original_freq_hz'] * 2.0
    detail['duplicated_freq_hz']   = np.nan
    detail['is_octave_duplicate']  = False
    detail['source_rank']          = np.nan
    detail['same_time_frame']      = True

    detail = detail.sort_values(['frame_idx', 'magnitude_db', 'freq_hz'], ascending=[True, False, True])
    detail['rank_in_block'] = detail.groupby('frame_idx', sort=False).cumcount() + 1
    detail['source_rank']   = detail['rank_in_block']

    detail = _enrich_musical_columns(detail)

    duplicate_rows = detail.loc[
        detail['freq_octave_up_hz'].notna() & (detail['freq_octave_up_hz'] <= NYQUIST)
    ].copy()
    if not duplicate_rows.empty:
        duplicate_rows['freq_hz']          = duplicate_rows['freq_octave_up_hz']
        duplicate_rows['duplicated_freq_hz']= duplicate_rows['freq_hz']
        duplicate_rows['freq_band']        = assign_frequency_band_vec(duplicate_rows['freq_hz'].to_numpy())
        duplicate_rows['source_band']      = duplicate_rows['freq_band']
        duplicate_rows['is_octave_duplicate'] = True
        duplicate_rows['source_rank']      = duplicate_rows['rank_in_block']
        duplicate_rows['same_time_frame']  = True
        duplicate_rows = _enrich_musical_columns(duplicate_rows)
        detail = pd.concat([detail, duplicate_rows], ignore_index=True, sort=False)

    detail = detail.sort_values(
        ['frame_idx', 'rank_in_block', 'is_octave_duplicate', 'freq_hz'],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)

    for column_name in base_columns:
        if column_name not in detail.columns:
            detail[column_name] = np.nan

    return detail


def build_block_summary_row(df_detail, song_id, block_idx, start_time, end_time, duration_sec,
                            valence=np.nan, arousal=np.nan, method=FINGERPRINT_METHOD,
                            title=None, artist=None, genre=None):
    duration_sec = float(duration_sec) if pd.notna(duration_sec) else float(max(float(end_time) - float(start_time), 0.0))
    quadrant, quadrant_label = classify_emotional_quadrant(valence, arousal)

    row = {
        'song_id': song_id, 'block_idx': int(block_idx),
        'start_time': float(start_time), 'end_time': float(end_time),
        'duration_sec': duration_sec,
        'valence': float(valence) if pd.notna(valence) else np.nan,
        'arousal': float(arousal) if pd.notna(arousal) else np.nan,
        'quadrant': quadrant, 'quadrant_label': quadrant_label,
        'method': method, 'title': title, 'artist': artist, 'genre': genre,
    }

    top_columns = [
        (f'fp_top_{r}_freq_hz', f'fp_top_{r}_magnitude_db',
         f'fp_top_{r}_midi', f'fp_top_{r}_octave', f'fp_top_{r}_semitone_approx')
        for r in range(1, TOP_K_RANK + 1)
    ]

    if df_detail is None or df_detail.empty:
        for column_group in top_columns:
            for column_name in column_group:
                row[column_name] = np.nan
        for col in ('fp_freq_mean', 'fp_freq_std', 'fp_mag_mean', 'fp_mag_std',
                    'fp_octdup_freq_mean', 'fp_octdup_freq_std', 'fp_octave_duplicate_count',
                    'fp_octave_mean', 'fp_octave_std', 'fp_midi_mean', 'fp_midi_std',
                    'fp_semitone_mean', 'fp_semitone_std', 'fp_unique_pitch_classes',
                    'fp_event_count', 'n_original_peaks', 'n_octave_duplicates', 'n_total_events'):
            row[col] = 0 if col in ('fp_octave_duplicate_count', 'fp_unique_pitch_classes',
                                    'fp_event_count', 'n_original_peaks', 'n_octave_duplicates', 'n_total_events') else np.nan
        return _add_block_engineered_features(row, source_events=None, duration_sec=duration_sec)

    events       = df_detail.loc[~df_detail['is_octave_duplicate']].copy()
    duplicates   = df_detail.loc[df_detail['is_octave_duplicate']].copy()
    source_events = events if not events.empty else df_detail.copy()

    freq_mean, freq_std       = _safe_mean_std(source_events['freq_hz'])
    mag_mean,  mag_std        = _safe_mean_std(source_events['magnitude_db'])
    dup_mean,  dup_std        = _safe_mean_std(duplicates['duplicated_freq_hz'])
    octave_mean, octave_std   = _safe_mean_std(source_events['octave'])
    midi_mean,  midi_std      = _safe_mean_std(source_events['midi'])
    semitone_mean, semitone_std = _safe_mean_std(source_events['semitone_approx'])

    row.update({
        'fp_freq_mean': freq_mean, 'fp_freq_std': freq_std,
        'fp_mag_mean': mag_mean,   'fp_mag_std': mag_std,
        'fp_octdup_freq_mean': dup_mean, 'fp_octdup_freq_std': dup_std,
        'fp_octave_duplicate_count': int(len(duplicates)),
        'fp_octave_mean': octave_mean, 'fp_octave_std': octave_std,
        'fp_midi_mean': midi_mean,     'fp_midi_std': midi_std,
        'fp_semitone_mean': semitone_mean, 'fp_semitone_std': semitone_std,
        'fp_unique_pitch_classes': int(source_events['pitch_class'].dropna().nunique()),
        'fp_event_count': int(len(source_events)),
    })

    ranked = (
        source_events.loc[
            source_events.groupby('freq_hz', sort=False)['magnitude_db'].idxmax()
        ]
        .sort_values('magnitude_db', ascending=False)
        .head(TOP_K_RANK)
        .reset_index(drop=True)
    )

    for rank_idx in range(1, TOP_K_RANK + 1):
        prefix = f'fp_top_{rank_idx}'
        if rank_idx <= len(ranked):
            ev = ranked.iloc[rank_idx - 1]
            row[f'{prefix}_freq_hz']        = float(ev['freq_hz'])
            row[f'{prefix}_magnitude_db']   = float(ev['magnitude_db'])
            row[f'{prefix}_midi']           = float(ev['midi'])           if pd.notna(ev['midi'])           else np.nan
            row[f'{prefix}_octave']         = float(ev['octave'])         if pd.notna(ev['octave'])         else np.nan
            row[f'{prefix}_semitone_approx']= float(ev['semitone_approx'])if pd.notna(ev['semitone_approx'])else np.nan
        else:
            for sfx in ('_freq_hz', '_magnitude_db', '_midi', '_octave', '_semitone_approx'):
                row[f'{prefix}{sfx}'] = np.nan

    row['n_original_peaks']  = int(len(source_events))
    row['n_octave_duplicates']= int(len(duplicates))
    row['n_total_events']     = int(len(df_detail))
    return _add_block_engineered_features(row, source_events=source_events, duration_sec=duration_sec)


print('Funcoes de construcao de DataFrames carregadas (vectorizadas)')


Funcoes de construcao de DataFrames carregadas (vectorizadas)


# ✨ 4. Geração do Fingerprint (Picos Harmônicos)

Algoritmo híbrido que combina o threshold por percentil com a seleção de picos (constellation map) para garantir densidade e relevância.

In [10]:
# ============================================================================
# DETECCAO DE PICOS — VECTORIZADA
# Substitui o loop Python por frame por operacoes numpy matriciais:
#   argpartition(-B_act, k, axis=0) seleciona top-k por coluna de uma vez.
# Elimina o gargalo: ~430 iteracoes Python/bloco x 4 bandas x 7208 blocos.
# ============================================================================


def _peaks_with_percentile(S_db, freqs, times, pctl):
    """Detecta picos por percentil em cada banda — implementacao vectorizada."""
    if S_db is None or getattr(S_db, 'size', 0) == 0:
        return np.array([]), np.array([]), np.array([])

    S_db  = np.asarray(S_db,  dtype=np.float32)
    freqs = np.asarray(freqs, dtype=np.float32)
    times = np.asarray(times, dtype=np.float32)

    if S_db.ndim != 2 or freqs.size != S_db.shape[0] or times.size != S_db.shape[1]:
        return np.array([]), np.array([]), np.array([])

    S_for = equalize_for_peaks(S_db) if EQUALIZE_FOR_PEAKS else S_db
    k     = int(TOPK_PER_BAND)

    pk_t_parts, pk_f_parts, pk_m_parts = [], [], []

    for lo, hi in BANDS:
        band_idx = np.flatnonzero((freqs >= lo) & (freqs < hi))
        if band_idx.size == 0:
            continue

        B = S_for[band_idx, :]   # [n_band_bins, n_frames]
        if B.size == 0:
            continue

        # Threshold por frame e mascara
        thr      = np.percentile(B, float(pctl), axis=0).astype(np.float32)  # [n_frames]
        B_masked = np.where(B >= thr[None, :], B, -np.inf).astype(np.float32)

        # Frames ativos: ao menos um bin acima do threshold
        active = np.flatnonzero(np.isfinite(B_masked).any(axis=0))
        if active.size == 0:
            continue

        B_act    = B_masked[:, active]   # [n_band_bins, n_active]
        actual_k = min(k, band_idx.size)
        n_active = active.size

        # Top-k bin indices por frame ativo — operacao matricial numpy
        if actual_k < band_idx.size:
            # np.argpartition(-B_act, actual_k, axis=0)[:actual_k] =
            # indices dos actual_k menores de -B_act = maiores de B_act
            local_top = np.argpartition(-B_act, actual_k - 1, axis=0)[:actual_k, :]
        else:
            local_top = np.tile(np.arange(band_idx.size)[:, None], (1, n_active))
        # local_top: [actual_k, n_active]

        rows_flat = local_top.ravel()                       # [actual_k * n_active]
        cols_flat = np.tile(np.arange(n_active), actual_k) # [actual_k * n_active]

        # Filtrar bins que estavam abaixo do threshold (-inf)
        vals_flat = B_act[rows_flat, cols_flat]
        valid     = np.isfinite(vals_flat)
        rows_flat = rows_flat[valid]
        cols_flat = cols_flat[valid]

        if rows_flat.size == 0:
            continue

        global_bin = band_idx[rows_flat]
        frame_abs  = active[cols_flat]

        # Magnitudes do S_db original (nao equalizado)
        pk_t_parts.append(times[frame_abs])
        pk_f_parts.append(freqs[global_bin])
        pk_m_parts.append(S_db[global_bin, frame_abs])

    if not pk_t_parts:
        return np.array([]), np.array([]), np.array([])

    return (
        np.concatenate(pk_t_parts).astype(np.float32),
        np.concatenate(pk_f_parts).astype(np.float32),
        np.concatenate(pk_m_parts).astype(np.float32),
    )


def estimate_peaks_auto_density(S_db, freqs, times, target_pps=TARGET_PEAKS_PER_SEC):
    if S_db is None or getattr(S_db, 'size', 0) == 0:
        return np.array([]), np.array([]), np.array([])
    dur = max(float(np.nanmax(times) - np.nanmin(times)), 1e-6)
    target_total = int(target_pps * dur)
    lo, hi = PCTL_SEARCH_RANGE
    best_pt, best_pf, best_pm = np.array([]), np.array([]), np.array([])
    best_diff = float('inf')
    for _ in range(MAX_BINSEARCH_ITERS):
        mid = (lo + hi) / 2.0
        pt, pf, pm = _peaks_with_percentile(S_db, freqs, times, pctl=mid)
        diff = abs(int(pt.size) - target_total)
        if diff < best_diff:
            best_diff, best_pt, best_pf, best_pm = diff, pt, pf, pm
        if target_total == 0 or abs(int(pt.size) - target_total) / max(target_total, 1) <= TOLERANCE_RELATIVE:
            break
        if int(pt.size) > target_total:
            lo = mid
        else:
            hi = mid
    return best_pt, best_pf, best_pm


print('Funcoes de deteccao de picos carregadas (vectorizadas)')


Funcoes de deteccao de picos carregadas (vectorizadas)


# 🖼️ 5. Estilização e Layout das Figuras

In [11]:
# Funcoes de analise desta etapa foram removidas por nao serem usadas no pipeline atual.
print("Sem funcoes de analise tabular auxiliares nesta versao")

Sem funcoes de analise tabular auxiliares nesta versao


In [12]:
# ============================================================================
# FUNCOES DE PERSISTENCIA POR MUSICA
# Estrutura: FINGERPRINT_DIR / song_XXX / <arquivos da musica>
# ============================================================================

def _song_tag(song_id):
    try:
        return f"song_{int(song_id):03d}"
    except Exception:
        return f"song_{str(song_id)}"


def _song_dir(song_id):
    """Retorna o diretorio compartilhado songs/ dentro de fingerprint_rank."""
    d = FINGERPRINT_DIR / "songs"
    d.mkdir(parents=True, exist_ok=True)
    return d




def drop_target_leakage_columns(df):
    """Preserva valence/arousal/quadrant/quadrant_label para treino,
    mas remove colunas derivadas diretamente do target que poderiam vazar informação
    caso fossem usadas acidentalmente como features.
    """
    if df is None or df.empty:
        return df if df is not None else pd.DataFrame()

    drop_cols = [c for c in globals().get("TARGET_LEAKAGE_COLUMNS", []) if c in df.columns]

    if globals().get("INCLUDE_TARGETS_IN_OUTPUT", True):
        # Mantém as colunas de rótulo, remove apenas derivadas de target, se existirem.
        return df.drop(columns=drop_cols, errors="ignore")

    # Modo estrito: remove também os rótulos emocionais.
    strict_cols = list(globals().get("TARGET_COLUMNS_TO_KEEP_IN_OUTPUT", [])) + drop_cols
    strict_cols = [c for c in strict_cols if c in df.columns]
    return df.drop(columns=strict_cols, errors="ignore")


def prepare_fingerprint_rank_dataframe(rows_or_df):
    """Aplica o mesmo schema usado no fingerprint_rank.parquet."""
    if rows_or_df is None:
        return pd.DataFrame()

    df_rank = rows_or_df.copy() if isinstance(rows_or_df, pd.DataFrame) else pd.DataFrame(rows_or_df)
    if df_rank.empty:
        return df_rank

    rename_map = {
        "block_idx":    "block_id",
        "start_time":   "block_start_sec",
        "end_time":     "block_end_sec",
        "duration_sec": "block_duration_sec",
        "fp_mag_mean":  "fp_magnitude_mean",
        "fp_mag_std":   "fp_magnitude_std",
    }
    df_rank = df_rank.rename(columns={k: v for k, v in rename_map.items() if k in df_rank.columns})
    df_rank = drop_target_leakage_columns(df_rank)

    id_cols     = ["song_id", "filename", "block_id", "block_start_sec", "block_end_sec", "block_duration_sec"]
    target_cols = ["valence", "arousal", "quadrant", "quadrant_label"]
    meta_cols   = ["title", "artist", "genre", "source_file", "method"]

    def _col_key(c):
        if c in id_cols:     return (0, id_cols.index(c), c)
        if c in target_cols: return (1, target_cols.index(c), c)
        if c.startswith("fp_top_"):              return (2, 0, c)
        if c.startswith("fp_magnitude"):         return (2, 1, c)
        if c.startswith("fp_freq"):              return (2, 2, c)
        if c.startswith("energy_db_"):           return (2, 3, c)
        if c.startswith("fp_"):                  return (2, 4, c)
        if c in meta_cols:   return (3, meta_cols.index(c), c)
        return (4, 0, c)

    ordered = sorted(df_rank.columns, key=_col_key)
    return df_rank[ordered]


def save_song_rank_parquet(song_rows, song_id):
    """Salva todos os blocos ja processados de um song_id em um Parquet proprio."""
    df_song_rank = prepare_fingerprint_rank_dataframe(song_rows)
    if df_song_rank.empty:
        return None

    tag = _song_tag(song_id)
    song_path = _song_dir(song_id) / f"{tag}_fingerprint_rank.parquet"
    df_song_rank.to_parquet(song_path, index=False, engine="pyarrow", compression="snappy")
    return song_path


def save_song_outputs(df_song_summary, df_song_detail, df_song_temporal, song_id):
    """Salva os Parquets principais da musica em fingerprint_rank/song_XXX/."""
    tag = _song_tag(song_id)
    d = _song_dir(song_id)

    summary_path = d / f"{tag}_blocks_summary.parquet"
    detail_path = d / f"{tag}_blocks_detail.parquet"
    temporal_path = d / f"{tag}_temporal_curves.parquet"

    if df_song_summary is None: df_song_summary = pd.DataFrame()
    if df_song_detail is None: df_song_detail = pd.DataFrame()
    if df_song_temporal is None: df_song_temporal = pd.DataFrame()

    df_song_summary = drop_target_leakage_columns(df_song_summary)
    df_song_detail = drop_target_leakage_columns(df_song_detail)
    df_song_temporal = drop_target_leakage_columns(df_song_temporal)

    df_song_summary.to_parquet(summary_path, index=False, engine="pyarrow", compression="snappy")

    detail_saved = False
    if SAVE_DETAIL_EVENTS and not df_song_detail.empty:
        df_song_detail.to_parquet(detail_path, index=False, engine="pyarrow", compression="snappy")
        detail_saved = True
    else:
        detail_path = None

    temporal_saved = False
    if SAVE_TEMPORAL_CURVES and not df_song_temporal.empty:
        df_song_temporal.to_parquet(temporal_path, index=False, engine="pyarrow", compression="snappy")
        temporal_saved = True
    else:
        temporal_path = None

    return str(summary_path), str(detail_path) if detail_saved else None, str(temporal_path) if temporal_saved else None


print("Funcoes de persistencia por musica carregadas")
print(f"     Estrutura: {FINGERPRINT_DIR}/song_XXX/<arquivos>")

Funcoes de persistencia por musica carregadas
     Estrutura: C:\dev\python\TCC Ajustado\Dados\fingerprint_rank/song_XXX/<arquivos>


# 📈 6. Análise Temporal e Extração por Banda

Extração das frequências evidentes e scores emocionais sincronizados em janelas de 500ms.

In [13]:
# Funcoes auxiliares legadas de banda foram removidas por nao uso direto.
print("Sem funcoes auxiliares de banda nesta versao")

Sem funcoes auxiliares de banda nesta versao


In [14]:
def _hz_label(v):
    if v is None or not np.isfinite(v):
        return "-"
    if v >= 1000:
        return f"{v / 1000.0:.1f}kHz"
    return f"{int(round(v))}Hz"

# 📊 7. Motor de Visualização

In [15]:
def plot_temporal_curves(df_temporal, out_dir, song_id, block_idx=None,
                         S_db=None, freqs=None, times=None, title_prefix='', export_png=False):
    """Monta o painel horizontal por bloco com espectrograma, top-k e V/A."""
    if df_temporal is None or df_temporal.empty:
        return None

    song_tag = f"song_{int(song_id):03d}" if pd.notna(song_id) else f"song_{str(song_id)}"
    song_dir = os.path.join(out_dir, song_tag)
    os.makedirs(song_dir, exist_ok=True)

    fmin_plot, fmax_plot = YLIM_HZ

    freqs_arr = np.asarray(freqs, dtype=float) if freqs is not None else np.array([])
    times_arr = np.asarray(times, dtype=float) if times is not None else np.array([])

    va_vals = []
    for col_va in ('valence', 'arousal'):
        if col_va in df_temporal.columns:
            va_vals.extend(df_temporal[col_va].dropna().tolist())
    if va_vals:
        va_min, va_max = min(va_vals), max(va_vals)
        va_span = max(va_max - va_min, 1e-6)
        va_pad = max(va_span * 0.25, 0.05)
        va_range = [max(-1.05, va_min - va_pad), min(1.05, va_max + va_pad)]
    else:
        va_range = [-1.1, 1.1]

    has_spec = (S_db is not None) and (np.asarray(S_db).size > 0)
    n_cols = 3 if has_spec else 2

    if n_cols == 3:
        col_widths = [0.28, 0.42, 0.30]
        spec_col, fp_col, va_col = 1, 2, 3
        titles = ('A. Espectrograma', 'B. Fingerprint Top-K', 'C. Dinamica Emocional')
    else:
        col_widths = [0.60, 0.40]
        fp_col, va_col = 1, 2
        titles = ('A. Fingerprint Top-K', 'B. Dinamica Emocional')

    fig = make_subplots(rows=1, cols=n_cols, column_widths=col_widths,
                        horizontal_spacing=0.06, subplot_titles=titles)

    # Painel do espectrograma
    if has_spec and freqs_arr.size and times_arr.size:
        S_plot = np.asarray(S_db, dtype=float)
        fig.add_trace(go.Heatmap(
            x=times_arr, y=freqs_arr, z=S_plot,
            colorscale='Viridis',
            colorbar=dict(title='dB', len=0.6, thickness=12, x=0.26),
            hovertemplate='t=%{x:.2f}s<br>f=%{y:.0f}Hz<br>db=%{z:.1f}<extra></extra>',
            showscale=True,
        ), row=1, col=spec_col)
        fig.update_yaxes(type='log', row=1, col=spec_col,
                         range=[math.log10(max(float(freqs_arr.min()), 1)), math.log10(float(freqs_arr.max()))],
                         title_text='Freq (Hz)')
        fig.update_xaxes(title_text='Tempo (s)', row=1, col=spec_col)

    # Faixas de frequencia no painel de fingerprint
    band_bg = [
        (max(fmin_plot, 20), min(fmax_plot, 250), 'rgba(31,119,180,0.05)'),
        (max(fmin_plot, 250), min(fmax_plot, 4000), 'rgba(255,127,14,0.04)'),
        (max(fmin_plot, 4000), fmax_plot, 'rgba(44,160,44,0.03)'),
    ]
    for f_lo, f_hi, color in band_bg:
        if f_lo >= f_hi:
            continue
        fig.add_hrect(y0=f_lo, y1=f_hi, fillcolor=color, line_width=0, row=1, col=fp_col)

    if px is not None:
        colors = px.colors.qualitative.Plotly
    else:
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                  '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

    for k in range(1, TOP_K_RANK + 1):
        col_name = f'top_{k}'
        if col_name not in df_temporal.columns:
            continue
        y_vals = df_temporal[col_name]
        hover_hz = [_hz_label(v) if pd.notna(v) else '-' for v in y_vals]
        is_top1 = (k == 1)
        width = 2.8 if is_top1 else 1.0
        opacity = 1.0 if is_top1 else max(0.08, 0.85 - k * 0.02)
        name = f'Top-{k}' if is_top1 else None
        fig.add_trace(go.Scatter(
            x=df_temporal['t_mid'], y=y_vals,
            name=name,
            mode='lines+markers' if is_top1 else 'lines',
            line=dict(color=colors[(k - 1) % len(colors)], width=width),
            marker=dict(size=6) if is_top1 else None,
            opacity=opacity,
            showlegend=is_top1,
            hovertemplate=f'<b>Top-{k}</b><br>t=%{{x:.2f}}s<br>f=%{{customdata}}<extra></extra>',
            customdata=hover_hz,
        ), row=1, col=fp_col)

    fig.update_yaxes(type='log', row=1, col=fp_col,
                     range=[math.log10(max(fmin_plot, 1)), math.log10(fmax_plot)],
                     title_text='Freq (Hz)')
    fig.update_xaxes(title_text='Tempo (s)', row=1, col=fp_col)

    if 'valence' in df_temporal.columns:
        fig.add_trace(go.Scatter(
            x=df_temporal['t_mid'], y=df_temporal['valence'],
            name='Valence', mode='lines+markers',
            line=dict(color='#d62728', width=2.4), marker=dict(size=5),
            fill='tozeroy', fillcolor='rgba(214,39,40,0.08)',
            hovertemplate='Valence: %{y:.3f}<br>t=%{x:.2f}s<extra></extra>',
        ), row=1, col=va_col)
    if 'arousal' in df_temporal.columns:
        fig.add_trace(go.Scatter(
            x=df_temporal['t_mid'], y=df_temporal['arousal'],
            name='Arousal', mode='lines+markers',
            line=dict(color='#9467bd', width=2.4, dash='dash'), marker=dict(size=5),
            hovertemplate='Arousal: %{y:.3f}<br>t=%{x:.2f}s<extra></extra>',
        ), row=1, col=va_col)

    fig.update_yaxes(title_text='Valence / Arousal', row=1, col=va_col, range=va_range)
    fig.update_xaxes(title_text='Tempo (s)', row=1, col=va_col)

    subtitle = f'Music {int(song_id):03d} - Block {int(block_idx):02d}' if block_idx is not None else f'Music {int(song_id):03d}'
    fig.update_layout(
        title=f'{title_prefix}{subtitle}',
        template='plotly_white',
        height=520, width=1600,
        hovermode='x unified',
        legend=dict(orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.02),
        margin=dict(l=60, r=160, t=90, b=60),
    )

    if block_idx is not None:
        fname = f'song_{int(song_id):03d}_block_{int(block_idx):02d}_temporal.html'
    else:
        fname = f'song_{int(song_id):03d}_temporal_analysis.html'
    out_file = os.path.join(song_dir, fname)
    fig.write_html(out_file, include_plotlyjs='cdn')

    if export_png and KALEIDO_AVAILABLE:
        try:
            png_name = out_file.replace('.html', '.png')
            fig.write_image(png_name, width=1600, height=520)
        except Exception as e:
            print(f"[WARN] Falha ao exportar PNG: {e}")

    return out_file

print('Funcoes de visualizacao por bloco carregadas')

Funcoes de visualizacao por bloco carregadas


## 7.1 Geração Completa por Bloco

In [16]:
def extract_block_peaks_and_temporal_features(
    row,
    S_db_pre=None,
    freqs_pre=None,
    times_pre=None,
    pt_pre=None,
    pf_pre=None,
    pm_pre=None,
    va_subset=None,
    generate_temporal_features=GENERATE_TEMPORAL_FEATURES,
):
    """Extrai picos, detalhe por evento e resumo do bloco sem fazer visualização."""
    song_id = row.get("song_id")
    block_idx = int(row.get("block_idx", 0))
    start_time = float(row.get("start_time", row.get("t_start", 0.0)))
    end_time = float(row.get("end_time", row.get("t_end", start_time)))
    duration_sec = float(row.get("duration_sec", max(end_time - start_time, 0.0)))
    valence = row.get("valence", np.nan)
    arousal = row.get("arousal", np.nan)
    method = row.get("method", FINGERPRINT_METHOD)
    title = row.get("title")
    artist = row.get("artist")
    genre = row.get("genre")

    quadrant, quadrant_label = classify_emotional_quadrant(valence, arousal)

    if S_db_pre is not None:
        S_db = np.asarray(S_db_pre)
        freqs = np.asarray(freqs_pre if freqs_pre is not None else [], dtype=float)
        times = np.asarray(times_pre if times_pre is not None else [], dtype=float)
    else:
        S_blk = bytes_to_ndarray(row.get("stft_bytes"))
        freqs = bytes_to_ndarray(row.get("freqs_bytes"))
        times = bytes_to_ndarray(row.get("times_bytes"))
        if S_blk.size == 0:
            S_db = np.empty((0, 0))
            freqs = np.asarray(freqs, dtype=float) if len(np.asarray(freqs).shape) else np.asarray([], dtype=float)
            times = np.asarray(times, dtype=float) if len(np.asarray(times).shape) else np.asarray([], dtype=float)
        else:
            S_db, freqs, times, _ = detect_scale_and_fix(S_blk, freqs, times)

    if pt_pre is not None:
        pt = np.asarray(pt_pre, dtype=float)
        pf = np.asarray(pf_pre, dtype=float)
        pm = np.asarray(pm_pre, dtype=float)
    else:
        if S_db.size == 0 or freqs.size == 0 or times.size == 0:
            pt = np.asarray([], dtype=float)
            pf = np.asarray([], dtype=float)
            pm = np.asarray([], dtype=float)
        else:
            pt, pf, pm = (
                estimate_peaks_auto_density(S_db, freqs, times)
                if AUTO_PEAK_DENSITY
                else _peaks_with_percentile(S_db, freqs, times, pctl=95.0)
            )

    df_detail = build_block_detail_dataframe(
        pt,
        pf,
        pm,
        song_id,
        block_idx,
        start_time,
        end_time,
        valence=valence,
        arousal=arousal,
        method=method,
        quadrant=quadrant,
        quadrant_label=quadrant_label,
    )
    df_block_summary = build_block_summary_row(
        df_detail,
        song_id,
        block_idx,
        start_time,
        end_time,
        duration_sec,
        valence=valence,
        arousal=arousal,
        method=method,
        title=title,
        artist=artist,
        genre=genre,
    )

    df_temporal = pd.DataFrame()
    if generate_temporal_features and S_db.size > 0 and freqs.size > 0 and times.size > 0:
        try:
            window_duration = 0.5
            window_starts = list(np.arange(start_time, end_time, window_duration))
            if not window_starts:
                window_starts = [start_time]

            temporal_rows = []
            source_detail = df_detail.loc[~df_detail["is_octave_duplicate"]].copy()

            for t_start_window in window_starts:
                t_end_window = min(t_start_window + window_duration, end_time)
                t_mid = (t_start_window + t_end_window) / 2.0

                if va_subset is not None and not va_subset.empty and "time_s" in va_subset.columns:
                    closest_idx = (va_subset["time_s"] - t_mid).abs().idxmin()
                    va_val = va_subset.loc[closest_idx, "valence"] if "valence" in va_subset.columns else valence
                    ar_val = va_subset.loc[closest_idx, "arousal"] if "arousal" in va_subset.columns else arousal
                else:
                    va_val = valence
                    ar_val = arousal

                temp_quadrant, temp_quadrant_label = classify_emotional_quadrant(va_val, ar_val)

                rows_in_window = source_detail[
                    (source_detail["time_sec"] >= t_start_window)
                    & (source_detail["time_sec"] < t_end_window)
                ].copy()

                row_temporal = {
                    "song_id": song_id,
                    "block_idx": block_idx,
                    "start_time": start_time,
                    "end_time": end_time,
                    "duration_sec": duration_sec,
                    "t_start": t_start_window,
                    "t_end": t_end_window,
                    "t_mid": t_mid,
                    "valence": va_val,
                    "arousal": ar_val,
                    "quadrant": temp_quadrant,
                    "quadrant_label": temp_quadrant_label,
                    "method": method,
                }

                if rows_in_window.empty:
                    for rank_idx in range(1, TOP_K_RANK + 1):
                        row_temporal[f"top_{rank_idx}"] = np.nan
                else:
                    ranked = rows_in_window.sort_values(
                        ["magnitude_db", "rank_in_block", "time_sec", "freq_hz"],
                        ascending=[False, True, True, True],
                    )
                    unique_freqs = []
                    for freq_value in ranked["freq_hz"].tolist():
                        if pd.isna(freq_value):
                            continue
                        if not any(np.isclose(freq_value, existing, atol=1e-6) for existing in unique_freqs):
                            unique_freqs.append(float(freq_value))
                        if len(unique_freqs) >= TOP_K_RANK:
                            break
                    for rank_idx in range(1, TOP_K_RANK + 1):
                        row_temporal[f"top_{rank_idx}"] = unique_freqs[rank_idx - 1] if rank_idx <= len(unique_freqs) else np.nan

                temporal_rows.append(row_temporal)

            df_temporal = pd.DataFrame(temporal_rows)
            for rank_idx in range(1, TOP_K_RANK + 1):
                col_abs = f"top_{rank_idx}"
                if col_abs not in df_temporal.columns:
                    df_temporal[col_abs] = np.nan
            ordered_columns = [
                "song_id",
                "block_idx",
                "start_time",
                "end_time",
                "duration_sec",
                "t_start",
                "t_end",
                "t_mid",
                "valence",
                "arousal",
                "quadrant",
                "quadrant_label",
                "method",
            ] + [f"top_{rank_idx}" for rank_idx in range(1, TOP_K_RANK + 1)]
            df_temporal = df_temporal[[col for col in ordered_columns if col in df_temporal.columns]]
        except Exception as e:
            print(f"[WARN] Erro ao gerar features temporais do bloco {block_idx}: {e}")

    return df_detail, df_block_summary, df_temporal


print("✅ Função de extração por bloco carregada")

✅ Função de extração por bloco carregada


# 🚀 8. Pipeline de Processamento (Execução)

STFT calculada localmente por bloco de áudio.
Magnitude normalizada pela referência global da música.
Blocos: `AUDIO_START_SEC` até o fim da música, passo `BLOCK_STEP_SEC`, descartando blocos < `MIN_BLOCK_DURATION_SEC`.
Valence e arousal alinhados pela média das amostras DEAM dentro do intervalo de cada bloco.
Saidas: `fingerprint_rank.parquet` consolidado e `song_XXX/song_XXX_fingerprint_rank.parquet` salvo ao final de cada musica.


In [17]:
# ============================================================================
# PIPELINE DE PROCESSAMENTO — LOOP POR MUSICA (otimizado)
# Otimizacoes:
#   1. DEAM_VA_BY_SONG: lookup O(1) por song_id
#   2. Dois passos: STFT dos blocos -> ref_song -> processar
#      (elimina STFT da musica inteira, que sozinha custava ~todos os blocos)
#   3. freqs e times pre-computados fora do loop
#   4. df_detail capturado por bloco -> raw_peaks acumulados como lista de dicts
#   5. Per-song prints suprimidos (Jupyter renderiza 5400+ linhas, muito lento)
# ============================================================================

all_features_block = []
raw_peaks_rows = []   # uma entrada por pico espectral; DataFrame criado em celula separada

audio_files = _iter_audio_files(IN_PATH)
if MAX_FILES_TO_PROCESS > 0:
    audio_files = audio_files[:MAX_FILES_TO_PROCESS]

print(f'Arquivos de audio encontrados: {len(audio_files)}')
print(f'Inicio={AUDIO_START_SEC}s | Tamanho={BLOCK_SIZE_SEC}s | Passo={BLOCK_STEP_SEC}s | Minimo={MIN_BLOCK_DURATION_SEC}s')
print()

iter_files = tqdm(audio_files, desc='Processando musicas') if tqdm is not None else audio_files

for audio_path, song_id in iter_files:
    try:
        song_id_int = int(song_id)
    except Exception:
        song_id_int = song_id

    filename = os.path.basename(audio_path)

    try:
        y, sr = librosa.load(audio_path, sr=(DOWNSAMPLE_AUDIO_TO or SR_TARGET), mono=True)
    except Exception as e:
        print(f'[ERR] Falha ao carregar {audio_path}: {e}')
        continue

    duration_total = float(len(y)) / sr
    if duration_total <= AUDIO_START_SEC:
        continue

    # Anotacoes V/A: lookup O(1) com pre-agrupamento
    va_song = None
    if DEAM_VA_BY_SONG is not None:
        try:
            va_song = DEAM_VA_BY_SONG.get_group(song_id_int)
        except KeyError:
            va_song = None

    # ------------------------------------------------------------------
    # PASSO 1: Coletar blocos validos e calcular STFT de cada bloco.
    # ref_song derivado do maximo entre blocos — sem STFT da musica inteira.
    # ------------------------------------------------------------------
    valid_blocks = []   # (t_start, t_end, act_dur, S_abs, valence_blk, arousal_blk)
    ref_song = 0.0

    t_block_start = AUDIO_START_SEC
    while t_block_start < duration_total:
        t_block_end     = t_block_start + BLOCK_SIZE_SEC
        actual_end      = min(t_block_end, duration_total)
        actual_duration = actual_end - t_block_start

        if actual_duration < MIN_BLOCK_DURATION_SEC:
            break

        block_start_sample = int(t_block_start * sr)
        block_end_sample   = int(actual_end * sr)
        y_block = y[block_start_sample:block_end_sample]

        if len(y_block) == 0:
            t_block_start += BLOCK_STEP_SEC
            continue

        try:
            S_complex_block = librosa.stft(
                y_block, n_fft=N_FFT_GLOBAL, hop_length=HOP_LENGTH_GLOBAL,
                window=WINDOW_TYPE, center=False,
            )
            S_abs_block = np.abs(S_complex_block).astype(np.float32)
            del S_complex_block
        except Exception as e:
            print(f'[WARN] STFT bloco song {song_id_int}: {e}')
            t_block_start += BLOCK_STEP_SEC
            continue

        block_max = float(np.max(S_abs_block))
        if block_max > ref_song:
            ref_song = block_max

        valence_blk = arousal_blk = np.nan
        if va_song is not None:
            va_win = va_song[
                (va_song['time_s'] >= t_block_start) &
                (va_song['time_s'] <  actual_end)
            ]
            if not va_win.empty:
                valence_blk = float(va_win['valence'].mean())
                arousal_blk = float(va_win['arousal'].mean())

        valid_blocks.append((t_block_start, actual_end, actual_duration, S_abs_block, valence_blk, arousal_blk))
        t_block_start += BLOCK_STEP_SEC

    if ref_song <= 0:
        ref_song = 1.0

    if not valid_blocks:
        continue

    # ------------------------------------------------------------------
    # PASSO 2: Processar blocos com ref_song conhecido
    # ------------------------------------------------------------------
    song_features_block = []
    song_block_count = 0

    for block_idx, (t_start, t_end, act_dur, S_abs_block, valence_blk, arousal_blk) in enumerate(valid_blocks):
        try:
            ref_block        = float(np.max(S_abs_block)) if np.max(S_abs_block) > 0 else 1.0
            block_norm_offset = (
                20.0 * np.log10(ref_song / ref_block)
                if ref_song > 0 and ref_block > 0 else 0.0
            )
            S_db_block = librosa.amplitude_to_db(S_abs_block, ref=ref_song)

            n_frames = S_db_block.shape[1]
            if n_frames == _FULL_BLOCK_FRAMES:
                times_block = _TIMES_LOCAL_FULL + t_start
            else:
                times_block = librosa.frames_to_time(
                    np.arange(n_frames), sr=sr, hop_length=HOP_LENGTH_GLOBAL
                ) + t_start
        except Exception as e:
            print(f'[WARN] Erro no bloco {block_idx} de song {song_id_int}: {e}')
            continue

        if S_db_block.size == 0 or times_block.size == 0:
            continue

        try:
            pt_blk, pf_blk, pm_blk = (
                estimate_peaks_auto_density(S_db_block, _FREQS_GLOBAL, times_block)
                if AUTO_PEAK_DENSITY
                else _peaks_with_percentile(S_db_block, _FREQS_GLOBAL, times_block, pctl=95.0)
            )
        except Exception as e:
            print(f'[WARN] Falha nos picos bloco {block_idx} song {song_id_int}: {e}')
            pt_blk = pf_blk = pm_blk = np.array([])

        row_dict = {
            'song_id':      song_id_int,
            'block_idx':    block_idx,
            'start_time':   t_start,
            'end_time':     t_end,
            't_start':      t_start,
            't_end':        t_end,
            'duration_sec': act_dur,
            'valence':      valence_blk,
            'arousal':      arousal_blk,
            'method':       FINGERPRINT_METHOD,
            'title': None, 'artist': None, 'genre': None,
        }

        try:
            # Captura df_detail para gerar raw_peaks
            df_detail, summary_dict, _ = extract_block_peaks_and_temporal_features(
                row=row_dict,
                S_db_pre=S_db_block, freqs_pre=_FREQS_GLOBAL, times_pre=times_block,
                pt_pre=pt_blk, pf_pre=pf_blk, pm_pre=pm_blk,
                va_subset=None, generate_temporal_features=False,
            )
        except Exception as e:
            print(f'[WARN] Erro bloco {block_idx} song {song_id_int}: {e}')
            continue

        for k_rank in range(1, TOP_K_RANK + 1):
            global_val = summary_dict.get(f'fp_top_{k_rank}_magnitude_db', np.nan)
            summary_dict[f'fp_top_{k_rank}_magnitude_norm_block'] = (
                float(global_val) + block_norm_offset
                if pd.notna(global_val) else np.nan
            )

        mag_mean_val = summary_dict.get('fp_mag_mean', np.nan)
        summary_dict['fp_magnitude_mean_norm_block'] = (
            float(mag_mean_val) + block_norm_offset
            if pd.notna(mag_mean_val) else np.nan
        )

        summary_dict['filename']    = filename
        summary_dict['source_file'] = audio_path
        all_features_block.append(summary_dict)
        song_features_block.append(summary_dict)
        song_block_count += 1

        # ------------------------------------------------------------------
        # Acumular raw_peaks: picos originais (nao-duplicados), top-TOP_K_RANK
        # por magnitude_db, uma entrada por pico por bloco.
        # AVISO: valence/arousal/quadrant_label mantidos apenas para
        # rastreabilidade. Remover antes de qualquer treinamento.
        # ------------------------------------------------------------------
        if df_detail is not None and not df_detail.empty:
            src_peaks = (
                df_detail.loc[~df_detail['is_octave_duplicate']]
                .sort_values('magnitude_db', ascending=False)
                .head(TOP_K_RANK)
                .reset_index(drop=True)
            )
            if not src_peaks.empty:
                pt_rp  = src_peaks['time_sec'].to_numpy(dtype=float)
                pf_rp  = src_peaks['freq_hz'].to_numpy(dtype=float)
                pm_rp  = src_peaks['magnitude_db'].to_numpy(dtype=float)
                mi_rp  = src_peaks['midi'].to_numpy(dtype=float)
                oc_rp  = src_peaks['octave'].to_numpy(dtype=float)
                se_rp  = src_peaks['semitone_approx'].to_numpy(dtype=float)
                pc_rp  = src_peaks['pitch_class'].to_numpy(dtype=float)
                bd_rp  = src_peaks['freq_band'].to_numpy(dtype=object)
                va_rp  = src_peaks['valence'].to_numpy(dtype=float)
                ar_rp  = src_peaks['arousal'].to_numpy(dtype=float)
                ql_rp  = src_peaks['quadrant_label'].to_numpy(dtype=object)
                n_rp   = len(src_peaks)

                for j in range(n_rp):
                    raw_peaks_rows.append({
                        'song_id':          int(song_id_int),
                        'filename':         filename,
                        'block_id':         int(block_idx),
                        'block_start_sec':  float(t_start),
                        'block_end_sec':    float(t_end),
                        'peak_rank_global': j + 1,
                        'peak_time_sec':    float(pt_rp[j]),
                        'frequency_hz':     float(pf_rp[j]),
                        'magnitude_db':     float(pm_rp[j]),
                        'midi_note':        float(mi_rp[j]) if np.isfinite(mi_rp[j]) else float('nan'),
                        'octave':           float(oc_rp[j]) if np.isfinite(oc_rp[j]) else float('nan'),
                        'semitone':         float(se_rp[j]) if np.isfinite(se_rp[j]) else float('nan'),
                        'pitch_class':      float(pc_rp[j]) if np.isfinite(pc_rp[j]) else float('nan'),
                        'freq_band':        str(bd_rp[j]),
                        'valence':          float(va_rp[j]) if np.isfinite(va_rp[j]) else float('nan'),
                        'arousal':          float(ar_rp[j]) if np.isfinite(ar_rp[j]) else float('nan'),
                        'quadrant_label':   str(ql_rp[j]),
                    })

    del valid_blocks
    save_song_rank_parquet(song_features_block, song_id_int)

print(f'\n[OK] Pipeline concluido!')
print(f'     Total de blocos   : {len(all_features_block)}')
print(f'     Total picos brutos: {len(raw_peaks_rows)}')


Arquivos de audio encontrados: 1802
Inicio=15.0s | Tamanho=10.0s | Passo=10.0s | Minimo=5.0s



Processando musicas:   0%|          | 0/1802 [00:00<?, ?it/s]


[OK] Pipeline concluido!
     Total de blocos   : 6506
     Total picos brutos: 195180


In [18]:
import os
import pandas as pd

# Garantir que o diretorio principal existe
if OUT_ROOT:
    os.makedirs(OUT_ROOT, exist_ok=True)

if not all_features_block:
    print("[WARN] Nenhum bloco processado. fingerprint_rank.parquet nao sera gerado.")
else:
    df_rank = prepare_fingerprint_rank_dataframe(all_features_block)

    # Salvar arquivo unico
    out_path = FINGERPRINT_DIR / OUTPUT_PARQUET_NAME
    df_rank.to_parquet(out_path, index=False, engine="pyarrow", compression="snappy")

    print(f"[OK] {OUTPUT_PARQUET_NAME} salvo!")
    print(f"     Shape  : {df_rank.shape[0]} blocos x {df_rank.shape[1]} colunas")
    print(f"     Songs  : {df_rank['song_id'].nunique() if 'song_id' in df_rank.columns else 'N/A'}")
    print(f"     Caminho: {out_path.absolute()}")

    try:
        display(df_rank.head(3))
    except Exception:
        print(df_rank.head(3).to_string())


[OK] fingerprint_rank.parquet salvo!
     Shape  : 6506 blocos x 222 colunas
     Songs  : 1802
     Caminho: C:\dev\python\TCC Ajustado\Dados\fingerprint_rank\fingerprint_rank.parquet


,song_id,filename,block_id,block_start_sec,block_end_sec,block_duration_sec,valence,arousal,quadrant,quadrant_label,...,fp_semitone_std,fp_unique_pitch_classes,title,artist,genre,source_file,method,n_octave_duplicates,n_original_peaks,n_total_events
0,10,10.mp3,0,15.0,25.0,10.0,0.099502,-0.158910,Q4,Calmo/Relaxado,...,19.487313,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,12916,21450,34366
1,10,10.mp3,1,25.0,35.0,10.0,0.035823,-0.174793,Q4,Calmo/Relaxado,...,18.065439,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,14972,21450,36422
2,10,10.mp3,2,35.0,45.0,10.0,-0.006201,-0.180349,Q3,Triste/Melancolico,...,18.515954,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,14509,21450,35959


In [19]:
# ============================================================================
# SALVAR fingerprint_rank_raw_peaks.parquet
# Uma linha por pico espectral original (is_octave_duplicate=False),
# top-TOP_K_RANK por magnitude_db por bloco.
#
# AVISO: valence/arousal/quadrant_label presentes apenas para rastreabilidade.
# Remover antes de qualquer treinamento supervisionado.
# Split treino/teste por song_id, NUNCA por linha.
# ============================================================================

RAW_PEAKS_NAME    = 'fingerprint_rank_raw_peaks.parquet'
raw_peaks_out_path = FINGERPRINT_DIR / RAW_PEAKS_NAME

if not raw_peaks_rows:
    print('[WARN] Nenhum pico bruto acumulado — fingerprint_rank_raw_peaks.parquet nao gerado.')
    df_rank_raw_peaks = pd.DataFrame()
else:
    df_rank_raw_peaks = pd.DataFrame(raw_peaks_rows)
    df_rank_raw_peaks['song_id']  = df_rank_raw_peaks['song_id'].astype(int)
    df_rank_raw_peaks['block_id'] = df_rank_raw_peaks['block_id'].astype(np.int32)

    # Otimizacao de tipos
    for col in ('freq_band', 'quadrant_label', 'filename'):
        if col in df_rank_raw_peaks.columns:
            df_rank_raw_peaks[col] = df_rank_raw_peaks[col].astype('category')
    for col in ('frequency_hz', 'magnitude_db', 'peak_time_sec',
                'midi_note', 'octave', 'semitone', 'pitch_class',
                'valence', 'arousal',
                'block_start_sec', 'block_end_sec'):
        if col in df_rank_raw_peaks.columns:
            df_rank_raw_peaks[col] = df_rank_raw_peaks[col].astype(np.float32)

    df_rank_raw_peaks = df_rank_raw_peaks.sort_values(
        ['song_id', 'block_id', 'peak_rank_global']
    ).reset_index(drop=True)

    # Ordem de colunas
    col_order = [
        'song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec',
        'peak_rank_global', 'peak_time_sec',
        'frequency_hz', 'magnitude_db',
        'midi_note', 'octave', 'semitone', 'pitch_class',
        'freq_band',
        'valence', 'arousal', 'quadrant_label',
    ]
    existing = [c for c in col_order if c in df_rank_raw_peaks.columns]
    extra    = [c for c in df_rank_raw_peaks.columns if c not in col_order]
    df_rank_raw_peaks = df_rank_raw_peaks[existing + extra]

    df_rank_raw_peaks.to_parquet(
        raw_peaks_out_path,
        index=False, engine='pyarrow', compression='snappy',
    )

    print(f'[OK] {RAW_PEAKS_NAME} salvo!')
    print(f'     Shape   : {df_rank_raw_peaks.shape[0]} picos x {df_rank_raw_peaks.shape[1]} colunas')
    print(f'     Songs   : {df_rank_raw_peaks["song_id"].nunique()}')
    print(f'     Blocos  : {df_rank_raw_peaks[["song_id","block_id"]].drop_duplicates().shape[0]}')
    print(f'     Picos/bloco (media): {df_rank_raw_peaks.groupby(["song_id","block_id"]).size().mean():.1f}')
    print(f'     Caminho : {raw_peaks_out_path.absolute()}')

    try:
        display(df_rank_raw_peaks.head(3))
    except Exception:
        print(df_rank_raw_peaks.head(3).to_string())


[OK] fingerprint_rank_raw_peaks.parquet salvo!
     Shape   : 195180 picos x 17 colunas
     Songs   : 1802
     Blocos  : 6506
     Picos/bloco (media): 30.0
     Caminho : C:\dev\python\TCC Ajustado\Dados\fingerprint_rank\fingerprint_rank_raw_peaks.parquet


,song_id,filename,block_id,block_start_sec,block_end_sec,peak_rank_global,peak_time_sec,frequency_hz,magnitude_db,midi_note,octave,semitone,pitch_class,freq_band,valence,arousal,quadrant_label
0,2,2.mp3,0,15.0,25.0,1,17.670296,193.798828,0.000000,54.804676,3.0,55.0,7.0,low,-0.08514,-0.137974,Triste/Melancolico
1,2,2.mp3,0,15.0,25.0,2,15.580499,193.798828,-1.374199,54.804676,3.0,55.0,7.0,low,-0.08514,-0.137974,Triste/Melancolico
2,2,2.mp3,0,15.0,25.0,3,22.453606,193.798828,-2.604416,54.804676,3.0,55.0,7.0,low,-0.08514,-0.137974,Triste/Melancolico


In [20]:
# ============================================================================
# VALIDACOES — fingerprint_rank_raw_peaks.parquet
# ============================================================================

if 'df_rank_raw_peaks' not in dir() or df_rank_raw_peaks.empty:
    print('[WARN] df_rank_raw_peaks vazio — validacoes ignoradas.')
else:
    required_cols = [
        'song_id', 'block_id', 'block_start_sec', 'block_end_sec',
        'peak_rank_global', 'peak_time_sec',
        'frequency_hz', 'magnitude_db',
        'midi_note', 'octave', 'semitone', 'pitch_class',
        'freq_band', 'valence', 'arousal', 'quadrant_label',
    ]
    missing = [c for c in required_cols if c not in df_rank_raw_peaks.columns]
    assert not missing, f'Colunas ausentes: {missing}'
    print('[OK] Todas as colunas obrigatorias presentes.')

    assert df_rank_raw_peaks[['song_id','block_id','frequency_hz','magnitude_db']].notna().all().all(), \
        'NaN em colunas criticas'
    print('[OK] Sem NaN em colunas criticas.')

    assert df_rank_raw_peaks['frequency_hz'].between(0, SR_TARGET / 2).all(), \
        'Frequencias fora de [0, Nyquist]'
    print('[OK] Frequencias dentro do range valido.')

    assert (df_rank_raw_peaks['peak_time_sec'] >= df_rank_raw_peaks['block_start_sec']).all(), \
        'peak_time_sec < block_start_sec'
    assert (df_rank_raw_peaks['peak_time_sec'] <  df_rank_raw_peaks['block_end_sec']).all(), \
        'peak_time_sec >= block_end_sec'
    print('[OK] Todos os picos dentro dos limites temporais dos seus blocos.')

    assert df_rank_raw_peaks['peak_rank_global'].between(1, TOP_K_RANK).all(), \
        f'peak_rank_global fora de [1, {TOP_K_RANK}]'
    print(f'[OK] peak_rank_global entre 1 e {TOP_K_RANK}.')

    # Sem sobreposicao entre blocos da mesma musica
    blocks_df = (
        df_rank_raw_peaks[['song_id','block_id','block_start_sec','block_end_sec']]
        .drop_duplicates()
        .sort_values(['song_id','block_start_sec'])
    )
    for _sid, _g in blocks_df.groupby('song_id'):
        _g = _g.sort_values('block_start_sec').reset_index(drop=True)
        if len(_g) > 1:
            assert (
                _g['block_start_sec'].iloc[1:].to_numpy()
                >= _g['block_end_sec'].iloc[:-1].to_numpy()
            ).all(), f'Sobreposicao de blocos para song_id={_sid}'
    print('[OK] Sem sobreposicao entre blocos.')

    print()
    print('--- Picos por banda ---')
    try:
        display(df_rank_raw_peaks.groupby('freq_band', observed=True).size())
    except Exception:
        print(df_rank_raw_peaks.groupby('freq_band', observed=True).size().to_string())

    print()
    print('[OK] Todas as validacoes passaram!')


[OK] Todas as colunas obrigatorias presentes.
[OK] Sem NaN em colunas criticas.
[OK] Frequencias dentro do range valido.
[OK] Todos os picos dentro dos limites temporais dos seus blocos.
[OK] peak_rank_global entre 1 e 30.
[OK] Sem sobreposicao entre blocos.

--- Picos por banda ---


freq_band
high           1300
low          130591
mid           63135
very_high       154
dtype: int64


[OK] Todas as validacoes passaram!


In [21]:
# ============================================================================
# VERIFICACAO DE SCHEMA — fingerprint_rank.parquet
# ============================================================================

import pathlib

out_path = FINGERPRINT_DIR / OUTPUT_PARQUET_NAME

if out_path.exists():
    df_check = pd.read_parquet(out_path)
    print(f"[OK] {OUTPUT_PARQUET_NAME} encontrado: shape={df_check.shape}")

    expected_cols = [
        "song_id", "filename", "block_id", "block_start_sec", "block_end_sec", "block_duration_sec",
        "valence", "arousal", "quadrant", "quadrant_label",
        "fp_top_1_freq_hz", "fp_top_1_magnitude_db",
        "fp_top_1_midi", "fp_top_1_octave", "fp_top_1_semitone_approx",
        "fp_magnitude_mean", "fp_magnitude_std",
        "fp_freq_mean", "fp_freq_std", "fp_midi_mean", "fp_midi_std",
        "fp_event_count", "fp_unique_pitch_classes",
    ]
    missing = [c for c in expected_cols if c not in df_check.columns]
    print(f"Schema: {len(expected_cols)-len(missing)}/{len(expected_cols)} colunas esperadas presentes")
    if missing:
        print(f"[WARN] Colunas faltantes: {missing}")

    print(f"\nColunas presentes ({len(df_check.columns)}):")
    for col in df_check.columns[:30]:
        print(f"  {col}: {df_check[col].dtype}")
    if len(df_check.columns) > 30:
        print(f"  ... e mais {len(df_check.columns)-30} colunas")

    try:
        display(df_check.head(3))
    except Exception:
        print(df_check.head(3).to_string())
else:
    print(f"[WARN] {OUTPUT_PARQUET_NAME} nao encontrado em {FINGERPRINT_DIR}")
    print("       Execute o pipeline (celula 26) e o salvamento (celula 27) primeiro.")

print("[OK] Verificacao de schema concluida")


[OK] fingerprint_rank.parquet encontrado: shape=(6506, 222)
Schema: 23/23 colunas esperadas presentes

Colunas presentes (222):
  song_id: int64
  filename: object
  block_id: int64
  block_start_sec: float64
  block_end_sec: float64
  block_duration_sec: float64
  valence: float64
  arousal: float64
  quadrant: object
  quadrant_label: object
  fp_top_10_freq_hz: float64
  fp_top_10_magnitude_db: float64
  fp_top_10_magnitude_norm_block: float64
  fp_top_10_midi: float64
  fp_top_10_octave: float64
  fp_top_10_semitone_approx: float64
  fp_top_11_freq_hz: float64
  fp_top_11_magnitude_db: float64
  fp_top_11_magnitude_norm_block: float64
  fp_top_11_midi: float64
  fp_top_11_octave: float64
  fp_top_11_semitone_approx: float64
  fp_top_12_freq_hz: float64
  fp_top_12_magnitude_db: float64
  fp_top_12_magnitude_norm_block: float64
  fp_top_12_midi: float64
  fp_top_12_octave: float64
  fp_top_12_semitone_approx: float64
  fp_top_13_freq_hz: float64
  fp_top_13_magnitude_db: float64
  .

,song_id,filename,block_id,block_start_sec,block_end_sec,block_duration_sec,valence,arousal,quadrant,quadrant_label,...,fp_semitone_std,fp_unique_pitch_classes,title,artist,genre,source_file,method,n_octave_duplicates,n_original_peaks,n_total_events
0,10,10.mp3,0,15.0,25.0,10.0,0.099502,-0.158910,Q4,Calmo/Relaxado,...,19.487313,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,12916,21450,34366
1,10,10.mp3,1,25.0,35.0,10.0,0.035823,-0.174793,Q4,Calmo/Relaxado,...,18.065439,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,14972,21450,36422
2,10,10.mp3,2,35.0,45.0,10.0,-0.006201,-0.180349,Q3,Triste/Melancolico,...,18.515954,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,14509,21450,35959


[OK] Verificacao de schema concluida


# 🗂️ 9. Consolidação — `fingerprint_rank.parquet`

O pipeline gera `fingerprint_rank.parquet` consolidado e, durante o processamento, salva tambem um Parquet por musica em `song_XXX/song_XXX_fingerprint_rank.parquet`, com:
- **1 linha = 1 música × 1 bloco de áudio**
- Blocos começando em `AUDIO_START_SEC` (15 s), com tamanho `BLOCK_SIZE_SEC` (10 s)
- STFT calculada localmente por bloco, magnitude em dB relativa à referência global da música
- Valence/Arousal = média das anotações DEAM no intervalo do bloco
- Blocos com duração < `MIN_BLOCK_DURATION_SEC` (5 s) são descartados


In [22]:

# ============================================================================
# FEATURES TEMPORAIS ENTRE BLOCOS
# ============================================================================
def add_block_delta_features(df):
    """Adiciona deltas e médias móveis por música no nível de bloco."""
    if df is None or df.empty:
        return df
    df = df.copy()
    sort_cols = [c for c in ["song_id", "block_idx", "start_time"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)

    candidates = [
        "fp_event_count", "n_original_peaks", "n_total_events",
        "fp_freq_mean", "fp_midi_mean", "fp_magnitude_mean",
        "fp_octave_mean", "fp_semitone_mean",
    ]
    candidates = [c for c in candidates if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

    if "song_id" not in df.columns:
        return df

    g = df.groupby("song_id", sort=False)
    for col in candidates:
        df[f"{col}_delta_prev"] = g[col].diff()
        df[f"{col}_roll3_mean"] = g[col].transform(lambda s: s.rolling(3, min_periods=1).mean())
        df[f"{col}_roll3_std"] = g[col].transform(lambda s: s.rolling(3, min_periods=1).std(ddof=0))

    return df

print("Função add_block_delta_features carregada")


Função add_block_delta_features carregada


In [23]:
# ============================================================================
# VERIFICACAO FINAL — fingerprint_rank.parquet
# O pipeline ja gera o arquivo unico consolidado diretamente.
# Nenhum arquivo intermediario e necessario.
# ============================================================================

import pathlib
import pandas as pd
import numpy as np

fp_root  = FINGERPRINT_DIR
out_path = fp_root / OUTPUT_PARQUET_NAME

if out_path.exists():
    df_final = pd.read_parquet(out_path)
    print(f"[OK] {OUTPUT_PARQUET_NAME} disponivel")
    print(f"     Shape  : {df_final.shape[0]} blocos x {df_final.shape[1]} colunas")
    print(f"     Songs  : {df_final['song_id'].nunique() if 'song_id' in df_final.columns else 'N/A'}")

    if "block_start_sec" in df_final.columns:
        print(f"     Primeiro bloco inicio: {df_final['block_start_sec'].min():.1f}s")
    if "block_end_sec" in df_final.columns:
        print(f"     Ultimo bloco fim     : {df_final['block_end_sec'].max():.1f}s")
    if "block_duration_sec" in df_final.columns:
        dur = df_final["block_duration_sec"]
        print(f"     Duracao blocos: min={dur.min():.1f}s | max={dur.max():.1f}s | media={dur.mean():.1f}s")

    try:
        display(df_final.head(5))
    except Exception:
        print(df_final.head(5).to_string())
else:
    print(f"[WARN] {OUTPUT_PARQUET_NAME} nao encontrado em {fp_root}")
    print("       Execute as celulas 26 (pipeline) e 27 (salvamento) para gerar o arquivo.")

print(f"\n[OK] Verificacao concluida! Resultados em: {fp_root.absolute()}")


[OK] fingerprint_rank.parquet disponivel
     Shape  : 6506 blocos x 222 colunas
     Songs  : 1802
     Primeiro bloco inicio: 15.0s
     Ultimo bloco fim     : 625.0s
     Duracao blocos: min=5.0s | max=10.0s | media=10.0s


,song_id,filename,block_id,block_start_sec,block_end_sec,block_duration_sec,valence,arousal,quadrant,quadrant_label,...,fp_semitone_std,fp_unique_pitch_classes,title,artist,genre,source_file,method,n_octave_duplicates,n_original_peaks,n_total_events
0,10,10.mp3,0,15.0,25.0,10.0,0.099502,-0.158910,Q4,Calmo/Relaxado,...,19.487313,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,12916,21450,34366
1,10,10.mp3,1,25.0,35.0,10.0,0.035823,-0.174793,Q4,Calmo/Relaxado,...,18.065439,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,14972,21450,36422
2,10,10.mp3,2,35.0,45.0,10.0,-0.006201,-0.180349,Q3,Triste/Melancolico,...,18.515954,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\10.mp3,top30_rank_absolute,14509,21450,35959
3,1000,1000.mp3,0,15.0,25.0,10.0,0.360038,0.452931,Q1,Alegre/Energetico,...,17.695375,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\1000.mp3,top30_rank_absolute,13383,21450,34833
4,1000,1000.mp3,1,25.0,35.0,10.0,0.334369,0.497053,Q1,Alegre/Energetico,...,17.740580,12,None,None,None,C:\dev\python\TCC Ajustado\Dados\audio\1000.mp3,top30_rank_absolute,14096,21450,35546



[OK] Verificacao concluida! Resultados em: C:\dev\python\TCC Ajustado\Dados\fingerprint_rank


In [24]:
# ============================================================================
# ENCERRAMENTO
# ============================================================================

out_path = FINGERPRINT_DIR / OUTPUT_PARQUET_NAME
if out_path.exists():
    df_final = pd.read_parquet(out_path)
    print("[OK] Experimento de fingerprinting concluido!")
    print(f"     Arquivo: {out_path.absolute()}")
    print(f"     Blocos : {len(df_final)}")
    print(f"     Songs  : {df_final['song_id'].nunique() if 'song_id' in df_final.columns else 'N/A'}")
    print(f"     Colunas: {len(df_final.columns)}")
else:
    print("[WARN] fingerprint_rank.parquet ainda nao gerado.")
    print("       Execute o pipeline de processamento para gerar o arquivo.")

print(f"Resultados em: {FINGERPRINT_DIR.absolute()}")


[OK] Experimento de fingerprinting concluido!
     Arquivo: C:\dev\python\TCC Ajustado\Dados\fingerprint_rank\fingerprint_rank.parquet
     Blocos : 6506
     Songs  : 1802
     Colunas: 222
Resultados em: C:\dev\python\TCC Ajustado\Dados\fingerprint_rank
